<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/Wan2.1_T2I_jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
!git clone https://github.com/city96/ComfyUI-GGUF /content/ComfyUI/custom_nodes/ComfyUI-GGUF
!pip install -q torchsde gguf

!apt install aria2 -qqy
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q3_K_S.gguf -d /content/ComfyUI/models/unet -o wan2.1-t2v-14b-Q3_K_M.gguf
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/umt5-xxl-encoder-gguf/resolve/main/umt5-xxl-encoder-Q3_K_S.gguf -d /content/ComfyUI/models/clip -o umt5-xxl-encoder-Q3_K_M.gguf
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d /content/ComfyUI/models/vae -o wan_2.1_vae.safetensors
!mkdir -p /content/ComfyUI/models/loras/FusionX
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/vrgamedevgirl84/Wan14BT2VFusioniX/resolve/main/FusionX_LoRa/Wan2.1_T2V_14B_FusionX_LoRA.safetensors -d /content/ComfyUI/models/loras/FusionX -o Wan2.1_T2V_14B_FusionX_LoRA.safetensors

/content
fatal: destination path '/content/ComfyUI' already exists and is not an empty directory.
fatal: destination path '/content/ComfyUI/custom_nodes/ComfyUI-GGUF' already exists and is not an empty directory.
aria2 is already the newest version (1.37.0+debian-1build3).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
271a54|OK  |       0B/s|/content/ComfyUI/models/unet/wan2.1-t2v-14b-Q3_K_M.gguf

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
342368|OK  |       0B/s|/content/ComfyUI/models/clip/umt5-xxl-encoder-Q3_K_M.gguf

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
3a0fd2|OK  |       0B/

In [36]:
%cd /content/ComfyUI

import torch
import random, time
import asyncio
from PIL import Image
import numpy as np

from nodes import NODE_CLASS_MAPPINGS, load_custom_node
import comfy_extras.nodes_model_advanced as nodes_model_advanced
import comfy_extras.nodes_hunyuan as nodes_hunyuan

# Load GGUF
await load_custom_node("/content/ComfyUI/custom_nodes/ComfyUI-GGUF")

UnetLoaderGGUF = NODE_CLASS_MAPPINGS["UnetLoaderGGUF"]()
CLIPLoaderGGUF = NODE_CLASS_MAPPINGS["CLIPLoaderGGUF"]()
LoraLoaderModelOnly = NODE_CLASS_MAPPINGS["LoraLoaderModelOnly"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()

CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()

KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
ModelSamplingSD3 = nodes_model_advanced.NODE_CLASS_MAPPINGS["ModelSamplingSD3"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()

with torch.inference_mode():
    unet = UnetLoaderGGUF.load_unet("wan2.1-t2v-14b-Q3_K_M.gguf")[0]
    clip = CLIPLoaderGGUF.load_clip("umt5-xxl-encoder-Q3_K_M.gguf", "wan")[0]
    lora = LoraLoaderModelOnly.load_lora_model_only(unet, "FusionX/Wan2.1_T2V_14B_FusionX_LoRA.safetensors", 1.0)[0]
    vae = VAELoader.load_vae("wan_2.1_vae.safetensors")[0]

/content/ComfyUI


In [40]:
with torch.inference_mode():
    seed = 0
    steps = 20
    cfg = 3.0
    sampler_name = "euler"
    scheduler = "simple"
    width = 720
    height = 1280
    shift = 5.0


    positive_prompt = "Her face and vagina in the shot, chubby latina thick thighs, Extreme close-up, low-angle perspective from between the legs of a naked Latina woman. Natural labia majora and minora, natural skin texture with visible pores and subtle imperfections, realistic pubic hair, slight natural moisture only on the genitals, realistic skin not plastic. The woman is holding her legs up, spreading wide, looking down at the camera with a seductive expression. Shallow depth of field, raw photograph, highly detailed anatomy, photorealistic skin texture"
    negative_prompt = "plastic skin, glossy skin, oily skin, waxy skin, airbrushed, over-smoothed, excessive shine, artificial wetness, cgi, 3d render, doll-like, smooth skin, bright red labia, cartoon vulva, artificial anatomy, 色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走"


    positive = CLIPTextEncode.encode(clip, positive_prompt)[0]
    negative = CLIPTextEncode.encode(clip, negative_prompt)[0]
    model = ModelSamplingSD3.patch(lora, shift)[0]
    latent_image = EmptyLatentImage.generate(width, height)[0]


    if seed == 0:
        random.seed(int(time.time()))
        seed = random.randint(0, 18446744073709551615)
